# 🔬 Computational Perceptual Holography Sandbox
### Project 2 — Interactive Degradation Explorer

Run each cell to explore how different degradations affect holographic reconstruction.
Tweak the parameters freely — that's the whole point.

**Modules:**
```
core/waveoptics.py   — ASM propagation, GS phase retrieval
core/degradation.py  — all degradation knobs
core/metrics.py      — PSNR, SSIM, LPIPS-proxy
core/scenes.py       — synthetic test scenes
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

from core.waveoptics  import gerchberg_saxton, reconstruct
from core.degradation import (
    degrade_resolution, quantise_phase,
    degrade_color, limit_viewing_angle,
    add_speckle, depth_planes_to_z_list,
)
from core.metrics import all_metrics
from core.scenes  import gaussian_spots, resolution_chart, letters, multi_depth_scene

print('All imports OK ✓')

## 1. Build a reference hologram

In [ ]:
# ── Parameters — change these freely ──────────────────────────────────────
SIZE       = 256      # 256 is fast; 512 is better quality (4× slower)
WAVELENGTH = 532e-9   # 532 nm green laser
DX         = 8e-6     # SLM pixel pitch 8 µm
Z          = 0.15     # reconstruction distance 15 cm
GS_ITER    = 30
# ──────────────────────────────────────────────────────────────────────────

target = gaussian_spots(SIZE, n_spots=5, sigma=SIZE*0.025, seed=42)
phase  = gerchberg_saxton(target, n_iter=GS_ITER, wavelength=WAVELENGTH, dx=DX, z=Z)
ref    = reconstruct(phase, wavelength=WAVELENGTH, dx=DX, z=Z)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(target, cmap='gray'); axes[0].set_title('Target scene'); axes[0].axis('off')
axes[1].imshow(phase,  cmap='hsv');  axes[1].set_title('Phase hologram'); axes[1].axis('off')
axes[2].imshow(ref,    cmap='gray'); axes[2].set_title('Reconstruction'); axes[2].axis('off')
plt.tight_layout(); plt.show()
print('Reference hologram built ✓')

## 2. Resolution degradation

In [ ]:
resolutions = [SIZE, SIZE//2, SIZE//4, SIZE//8]

fig, axes = plt.subplots(1, len(resolutions), figsize=(4*len(resolutions), 4))
for ax, res in zip(axes, resolutions):
    deg_phase = degrade_resolution(phase, res)
    recon     = reconstruct(deg_phase, wavelength=WAVELENGTH, dx=DX, z=Z)
    m = all_metrics(ref, recon)
    ax.imshow(recon, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{res}×{res}\nPSNR={m["psnr"]:.1f} SSIM={m["ssim"]:.3f}', fontsize=9)
    ax.axis('off')
plt.suptitle('Resolution Sweep', fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Phase quantisation

In [ ]:
bit_levels = [8, 4, 2, 1]

fig, axes = plt.subplots(1, len(bit_levels), figsize=(4*len(bit_levels), 4))
for ax, bits in zip(axes, bit_levels):
    q_phase = quantise_phase(phase, bits=bits)
    recon   = reconstruct(q_phase, wavelength=WAVELENGTH, dx=DX, z=Z)
    m = all_metrics(ref, recon)
    ax.imshow(recon, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{bits}-bit ({2**bits} levels)\nPSNR={m["psnr"]:.1f}', fontsize=9)
    ax.axis('off')
plt.suptitle('Phase Quantisation Sweep', fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Viewing angle (bandwidth)

In [ ]:
fractions = [1.0, 0.75, 0.5, 0.25, 0.1]

fig, axes = plt.subplots(1, len(fractions), figsize=(4*len(fractions), 4))
for ax, frac in zip(axes, fractions):
    lim_phase = limit_viewing_angle(phase, bandwidth_fraction=frac)
    recon     = reconstruct(lim_phase, wavelength=WAVELENGTH, dx=DX, z=Z)
    m = all_metrics(ref, recon)
    ax.imshow(recon, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'BW {int(frac*100)}%\nSSIM={m["ssim"]:.3f}', fontsize=9)
    ax.axis('off')
plt.suptitle('Viewing Angle / Bandwidth Sweep', fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Depth planes

In [ ]:
plane_counts = [4, 2, 1]
fig, axes = plt.subplots(1, len(plane_counts), figsize=(5*len(plane_counts), 4))

for ax, n_planes in zip(axes, plane_counts):
    layers = multi_depth_scene(SIZE, n_planes=n_planes)
    z_list = depth_planes_to_z_list(n_planes, z_near=0.08, z_far=0.25)
    combined_phase = np.zeros((SIZE, SIZE), dtype=np.float32)
    for layer, z_layer in zip(layers, z_list):
        lp = gerchberg_saxton(layer, n_iter=GS_ITER, wavelength=WAVELENGTH, dx=DX, z=z_layer)
        combined_phase += lp
    combined_phase = np.angle(np.exp(1j * combined_phase))
    z_mid = (z_list[0] + z_list[-1]) / 2
    recon = reconstruct(combined_phase, wavelength=WAVELENGTH, dx=DX, z=z_mid)
    ax.imshow(recon, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{n_planes} depth plane(s)', fontsize=10)
    ax.axis('off')
plt.suptitle('Depth Planes Sweep', fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Speckle noise

In [ ]:
sigmas = [0.0, 0.05, 0.1, 0.2, 0.4]
fig, axes = plt.subplots(1, len(sigmas), figsize=(4*len(sigmas), 4))
for ax, sigma in zip(axes, sigmas):
    noisy = add_speckle(ref, sigma=sigma) if sigma > 0 else ref.copy()
    m = all_metrics(ref, noisy)
    ax.imshow(noisy, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'σ={sigma:.2f}\nPSNR={m["psnr"]:.1f}', fontsize=9)
    ax.axis('off')
plt.suptitle('Speckle Noise Sweep', fontweight='bold')
plt.tight_layout(); plt.show()

## 7. The key insight — PSNR vs SSIM

Run this cell to see the perceptual-vs-physical metric discrepancy.
This is the core observation for the paper.

In [ ]:
import pandas as pd

results = []
experiments = [
    ('Resolution 128', degrade_resolution(phase, SIZE//2)),
    ('Resolution 64',  degrade_resolution(phase, SIZE//4)),
    ('4-bit phase',    quantise_phase(phase, 4)),
    ('2-bit phase',    quantise_phase(phase, 2)),
    ('1-bit phase',    quantise_phase(phase, 1)),
    ('BW 50%',         limit_viewing_angle(phase, 0.5)),
    ('BW 25%',         limit_viewing_angle(phase, 0.25)),
    ('BW 10%',         limit_viewing_angle(phase, 0.1)),
]

for name, deg_phase in experiments:
    recon = reconstruct(deg_phase, wavelength=WAVELENGTH, dx=DX, z=Z)
    m = all_metrics(ref, recon)
    m['name'] = name
    results.append(m)

df = pd.DataFrame(results).set_index('name')[['psnr','ssim','lpips_proxy']]
df = df.sort_values('psnr', ascending=False)
print(df.to_string())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
df['psnr'].plot(kind='barh', ax=ax1, color='#2563eb', alpha=0.8)
ax1.set_xlabel('PSNR (dB)'); ax1.set_title('Physical metric (PSNR)'); ax1.invert_yaxis()

df['ssim'].plot(kind='barh', ax=ax2, color='#16a34a', alpha=0.8)
ax2.set_xlabel('SSIM'); ax2.set_title('Perceptual metric (SSIM)'); ax2.invert_yaxis()

plt.suptitle('Physical vs Perceptual Metrics — same degradations, different story',
             fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()
print('\n→ Look for cases where PSNR ranking ≠ SSIM ranking.')
print('→ Those are the perceptually interesting cases for your paper.')